In [1]:
import numpy as np
import pandas as pd
import random
import uuid

# =========================
# CONFIG
# =========================
N_TRACKS = 500
POINTS_PER_TRACK = 20
N_REPORTS = 200

np.random.seed(42)
random.seed(42)

# =========================
# 1. MOCK SENSOR DATA
# =========================
sensor_rows = []

for tid in range(N_TRACKS):
    lat = np.random.uniform(10, 25)
    lon = np.random.uniform(100, 120)

    for t in range(POINTS_PER_TRACK):
        lat += np.random.normal(0, 0.05)
        lon += np.random.normal(0, 0.05)

        sensor_rows.append({
            "trackId": tid,
            "timestamp": t,
            "trackName": f"T{tid}",
            "mode3A": random.randint(1000, 9999),
            "latitude": lat,
            "longitude": lon,
            "altitude": np.random.uniform(8000, 12000),
            "heading": np.random.uniform(0, 360),
            "speed": np.random.uniform(200, 900),
        })

sensor_df = pd.DataFrame(sensor_rows)

# =========================
# 2. MOCK REPORT DATA
# =========================
event_rows = []

# map để biết ground truth (optional)
gt_matches = {}

track_groups = sensor_df.groupby("trackId")

for rid in range(N_REPORTS):
    # chọn ngẫu nhiên 1 track để tạo report
    tid = random.randint(0, N_TRACKS - 1)
    track = track_groups.get_group(tid).sort_values("timestamp")

    coords = track[["latitude", "longitude"]].values

    # lấy sub-trajectory (report chỉ là 1 đoạn)
    start = random.randint(0, len(coords) - 5)
    length = random.randint(3, 6)
    segment = coords[start:start+length]

    # thêm noise (quan trọng)
    noise = np.random.normal(0, 0.02, segment.shape)
    segment_noisy = segment + noise

    lats = segment_noisy[:, 0].tolist()
    lons = segment_noisy[:, 1].tolist()

    event_rows.append({
        "eventMention": f"Aircraft spotted near zone {rid}",
        "eventType": random.choice(["PATROL", "SURVEILLANCE", "UNKNOWN"]),
        "obj": f"Aircraft_{tid}",
        "nation": random.choice(["US", "CN", "RU", "VN"]),
        "objType": random.choice(["FIGHTER", "BOMBER", "DRONE"]),
        "quantity": random.randint(1, 4),
        "locsId": str(uuid.uuid4()),
        "locs": f"Zone_{random.randint(1,100)}",
        "coors.Type": "LineString",
        "coors.properties.basesName": f"Base_{random.randint(1,10)}",
        "coors.geometry.type": "LineString",
        "coors.longitudes": lons,
        "coors.latitudes": lats
    })

    # lưu ground truth (optional)
    gt_matches[rid] = tid

event_df = pd.DataFrame(event_rows)

# =========================
# 3. SAVE PARQUET
# =========================
sensor_df.to_parquet("sensor.parquet", index=False)
event_df.to_parquet("event.parquet", index=False)

print("✅ Saved:")
print(" - sensor.parquet")
print(" - event.parquet")

# =========================
# 4. (OPTIONAL) SAVE GROUND TRUTH
# =========================
gt_df = pd.DataFrame([
    {"report_id": rid, "track_id": tid}
    for rid, tid in gt_matches.items()
])

gt_df.to_parquet("ground_truth.parquet", index=False)

print(" - ground_truth.parquet")

# =========================
# 5. QUICK CHECK
# =========================
print("\nSensor sample:")
print(sensor_df.head())

print("\nEvent sample:")
print(event_df.head())

print("\nGT sample:")
print(gt_df.head())

✅ Saved:
 - sensor.parquet
 - event.parquet
 - ground_truth.parquet

Sensor sample:
   trackId  timestamp trackName  mode3A   latitude   longitude      altitude  \
0        0          0        T0    2824  15.650486  119.090438   8624.074562   
1        0          1        T0    1409  15.664438  119.140963  10832.290311   
2        0          2        T0    5506  15.640965  119.168091   8727.299869   
3        0          3        T0    5012  15.510337  119.215610   9164.916561   
4        0          4        T0    4657  15.464936  119.144995   9824.279937   

      heading       speed  
0   56.158027  240.658529  
1    7.410418  878.936897  
2   66.025624  412.969570  
3  220.267042  297.645702  
4  282.663346  339.771648  

Event sample:
                   eventMention     eventType           obj nation  objType  \
0  Aircraft spotted near zone 0  SURVEILLANCE  Aircraft_478     US    DRONE   
1  Aircraft spotted near zone 1        PATROL  Aircraft_369     VN    DRONE   
2  Aircraft spo

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [3]:
def build_trajectories(sensor_df):
    sensor_df = sensor_df.sort_values(['trackId', 'timestamp'])

    trajs = {}
    for tid, df in sensor_df.groupby('trackId'):
        trajs[tid] = df[['latitude','longitude','altitude','speed','heading','timestamp']].values

    return trajs

In [4]:
def split_sub_trajectories(trajs, min_len=4, max_len=8, stride=2):
    subs = []

    for tid, traj in trajs.items():
        n = len(traj)

        for w in range(min_len, max_len+1):
            for i in range(0, n-w+1, stride):
                seg = traj[i:i+w]
                subs.append({
                    "track_id": tid,
                    "segment": seg,
                    "t_start": seg[0, -1],
                    "t_end": seg[-1, -1]
                })

    return subs

In [5]:
def encode_subtrack(seg):
    xy = seg[:, :2]
    speed = seg[:, 3]
    heading = seg[:, 4]

    start = xy[0]
    end = xy[-1]

    vec = end - start
    length = np.linalg.norm(vec) + 1e-6
    direction = vec / length

    # curvature
    diffs = np.diff(xy, axis=0)
    angles = np.arctan2(diffs[:,1], diffs[:,0])
    curvature = np.std(angles)

    speed_mean = np.mean(speed)
    speed_std = np.std(speed)

    heading_vec = [
        np.cos(np.deg2rad(np.mean(heading))),
        np.sin(np.deg2rad(np.mean(heading)))
    ]

    return np.array([
        start[0], start[1],
        end[0], end[1],
        direction[0], direction[1],
        length,
        curvature,
        speed_mean,
        speed_std,
        heading_vec[0],
        heading_vec[1]
    ], dtype=np.float32)

In [6]:
def safe_array(x):
    if isinstance(x, list) or isinstance(x, np.ndarray):
        return np.array(x)
    try:
        return np.array(eval(x))
    except:
        return np.array([])

def encode_report(row, k=5):
    lons = safe_array(row['coors.longitudes'])
    lats = safe_array(row['coors.latitudes'])

    if len(lons) < k:
        lons = np.pad(lons, (0, k-len(lons)))
        lats = np.pad(lats, (0, k-len(lats)))

    idx = np.linspace(0, len(lons)-1, k).astype(int)
    pts = np.stack([lats[idx], lons[idx]], axis=1)

    diffs = np.diff(pts, axis=0)
    curvature = np.std(np.arctan2(diffs[:,1], diffs[:,0]))

    return np.concatenate([pts.flatten(), [curvature]]).astype(np.float32)

In [7]:
def build_features(sub_tracks, event_df):
    sub_vecs = np.stack([encode_subtrack(s["segment"]) for s in sub_tracks])
    report_vecs = np.stack(event_df.apply(encode_report, axis=1))

    sub_vecs = (sub_vecs - sub_vecs.mean(0)) / (sub_vecs.std(0)+1e-6)
    report_vecs = (report_vecs - report_vecs.mean(0)) / (report_vecs.std(0)+1e-6)

    return sub_vecs, report_vecs

In [8]:
def filter_candidates(sub_vecs, report_vecs, radius=0.05, max_k=10):
    candidates = []

    for s in sub_vecs:
        d = np.linalg.norm(report_vecs[:, :2] - s[:2], axis=1)
        idx = np.where(d < radius)[0]

        if len(idx) == 0:
            idx = np.argsort(d)[:max_k]

        candidates.append(idx)

    return candidates

In [9]:
def create_pairs(sub_vecs, report_vecs, candidates, k_pos=2, k_neg=4):
    pairs, labels = [], []

    for i, cand in enumerate(candidates):
        d = np.linalg.norm(report_vecs[cand, :2] - sub_vecs[i][:2], axis=1)
        order = cand[np.argsort(d)]

        pos = order[:k_pos]
        neg = order[k_pos:k_pos+k_neg]

        for p in pos:
            pairs.append((i, p))
            labels.append(1)

        for n in neg:
            pairs.append((i, n))
            labels.append(0)

    return np.array(pairs), np.array(labels)

In [10]:
class SubTrackDataset(Dataset):
    def __init__(self, pairs, labels, sub_vecs, report_vecs):
        self.pairs = pairs
        self.labels = labels
        self.sub_vecs = sub_vecs
        self.report_vecs = report_vecs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        i, j = self.pairs[idx]
        return (
            torch.tensor(self.sub_vecs[i]),
            torch.tensor(self.report_vecs[j]),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

In [11]:
class EmbeddingNet(nn.Module):
    def __init__(self, in_dim, emb_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=1)

class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, e1, e2, y):
        d = F.pairwise_distance(e1, e2)
        return (y*d**2 + (1-y)*F.relu(self.margin-d)**2).mean()

In [23]:
def train_model(sub_vecs, report_vecs, pairs, labels, epochs=10):
    dataset = SubTrackDataset(pairs, labels, sub_vecs, report_vecs)
    loader = DataLoader(dataset, batch_size=128, shuffle=True)

    net_t = EmbeddingNet(sub_vecs.shape[1])
    net_r = EmbeddingNet(report_vecs.shape[1])

    optimizer = torch.optim.Adam(
        list(net_t.parameters()) + list(net_r.parameters()),
        lr=1e-3
    )

    loss_fn = ContrastiveLoss()

    for ep in range(epochs):
        total = 0
        for t, r, y in loader:
            optimizer.zero_grad()
            loss = loss_fn(net_t(t), net_r(r), y)
            loss.backward()
            optimizer.step()
            total += loss.item()

        print(f"Epoch {ep+1}: loss={total/len(loader):.4f}")

    return net_t, net_r

In [13]:
def match_tracks(net_t, net_r, sub_tracks, sub_vecs, report_vecs):
    with torch.no_grad():
        e_t = net_t(torch.tensor(sub_vecs))
        e_r = net_r(torch.tensor(report_vecs))

        dist = torch.cdist(e_t, e_r)

    track_best = {}

    for i, s in enumerate(sub_tracks):
        tid = s["track_id"]

        best_report = dist[i].argmin().item()
        best_score = dist[i].min().item()

        if tid not in track_best or best_score < track_best[tid]["score"]:
            track_best[tid] = {
                "report_id": best_report,
                "score": best_score,
                "sub_idx": i
            }

    return track_best

In [22]:
def run_pipeline(sensor_df, event_df):
    trajs = build_trajectories(sensor_df)

    sub_tracks = split_sub_trajectories(trajs)

    sub_vecs, report_vecs = build_features(sub_tracks, event_df)

    candidates = filter_candidates(sub_vecs, report_vecs)

    pairs, labels = create_pairs(sub_vecs, report_vecs, candidates)

    net_t, net_r = train_model(sub_vecs, report_vecs, pairs, labels)

    results = match_tracks(net_t, net_r, sub_tracks, sub_vecs, report_vecs)

    return sub_tracks, results

In [15]:
sensor_df = pd.read_parquet("sensor.parquet")
event_df = pd.read_parquet("event.parquet")

In [16]:
#sensor_df = sensor_df.sample(5000)
#event_df = event_df.sample(500)

In [24]:
sub_tracks, results = run_pipeline(sensor_df, event_df)

Epoch 1: loss=0.2246
Epoch 2: loss=0.2136
Epoch 3: loss=0.2086
Epoch 4: loss=0.2049
Epoch 5: loss=0.2023
Epoch 6: loss=0.1995
Epoch 7: loss=0.1965
Epoch 8: loss=0.1934
Epoch 9: loss=0.1894
Epoch 10: loss=0.1862


In [18]:
df_results = pd.DataFrame([
    {
        "track_id": tid,
        "report_id": v["report_id"],
        "score": v["score"],
        "sub_idx": v["sub_idx"]
    }
    for tid, v in results.items()
])

df_results.to_csv("match_results_full.csv", index=False)

print("✅ Saved to match_results_full.csv")

✅ Saved to match_results_full.csv


In [26]:
import folium
import numpy as np

def plot_matches(sensor_df, event_df, sub_tracks, results, num_samples=10):
    # lấy center map
    center_lat = sensor_df["latitude"].mean()
    center_lon = sensor_df["longitude"].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

    # chọn 1 số track để vẽ (tránh lag)
    items = list(results.items())[:num_samples]

    for tid, v in items:
        rid = v["report_id"]
        score = v["score"]

        # ===== TRACK =====
        track_df = sensor_df[sensor_df["trackId"] == tid].sort_values("timestamp")

        track_coords = track_df[["latitude","longitude"]].values.tolist()

        folium.PolyLine(
            track_coords,
            color="blue",
            weight=3,
            opacity=0.7,
            tooltip=f"Track {tid} | score={score:.3f}"
        ).add_to(m)

        # ===== REPORT =====
        row = event_df.iloc[rid]

        lons = row["coors.longitudes"]
        lats = row["coors.latitudes"]

        # parse nếu là string
        if isinstance(lons, str):
            lons = eval(lons)
        if isinstance(lats, str):
            lats = eval(lats)

        report_coords = list(zip(lats, lons))

        folium.PolyLine(
            report_coords,
            color="red",
            weight=4,
            opacity=0.8,
            tooltip=f"Report {rid}"
        ).add_to(m)

        # ===== MARK START POINT =====
        if len(track_coords) > 0:
            folium.CircleMarker(
                track_coords[0],
                radius=4,
                color="blue",
                fill=True
            ).add_to(m)

        if len(report_coords) > 0:
            folium.CircleMarker(
                report_coords[0],
                radius=4,
                color="red",
                fill=True
            ).add_to(m)
        
        sub_idx = v["sub_idx"]
        seg = sub_tracks[sub_idx]["segment"]

        sub_coords = seg[:, :2].tolist()

        folium.PolyLine(
                 sub_coords,
                 color="green",
                 weight=5,
                 opacity=1.0,
                 tooltip="Matched segment"
                ).add_to(m)

    return m

In [29]:
m = plot_matches(sensor_df, event_df, sub_tracks, results, num_samples=200)
m